# notebook 03 — exploratory data analysis

look for patterns and trends in the cleaned data using charts. 8 visualisations covering revenue trends, top countries, top products, cancellations, and time patterns. each chart has a business insight below it.

**input:** data/processed/cleaned_sales.csv  
**output:** none (charts only)

In [ ]:
# Import libraries and load data
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load data
df = pd.read_csv('../data/processed/cleaned_sales.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Load cancellations
cancellations_df = pd.read_csv('../data/processed/cancellations.csv')
cancellations_df['InvoiceDate'] = pd.to_datetime(cancellations_df['InvoiceDate'])


In [ ]:
# 1. Monthly Revenue Trend
monthly_revenue = df.set_index('InvoiceDate').resample('ME')['Revenue'].sum().reset_index()

plt.figure(figsize=(12, 6))
plt.plot(monthly_revenue['InvoiceDate'], monthly_revenue['Revenue'], marker='o', linewidth=2, color='b')
plt.title('Monthly Revenue Trend (2010 - 2011)', fontsize=14)
plt.xlabel('Month', fontsize=12)
plt.ylabel('Total Revenue (£)', fontsize=12)
plt.axvline(pd.to_datetime('2011-11-01'), color='r', linestyle='--', label='Peak - Nov 2011')
plt.legend()
plt.tight_layout()
plt.show()


**Business Insight:** 
The chart shows a significant spike in revenue during November 2011, indicating strong seasonality driven by Q4 holiday shopping (Black Friday, early Christmas purchases). The slow season appears to be the first half of the year (Q1-Q2). For the business, this means inventory and marketing efforts must be heavily front-loaded ahead of Q4 to maximize the year's primary revenue opportunity.


In [ ]:
# 2. Revenue by Country — Top 10
country_revenue = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=country_revenue.values, y=country_revenue.index, palette='viridis')
plt.xscale('log')
plt.title('Top 10 Countries by Revenue (Log Scale)', fontsize=14)
plt.xlabel('Total Revenue (£) [Log Scale]', fontsize=12)
plt.ylabel('Country', fontsize=12)
plt.tight_layout()
plt.show()


**Business Insight:** 
The business is massively dependent on the United Kingdom, which dominates the revenue stream (we use a log scale here just to make the other countries visible). While the UK is the core market, this heavy reliance presents a concentration risk. There is a substantial international opportunity in nearby European countries like the Netherlands, EIRE, and Germany, where localized marketing could drive significant growth.


In [ ]:
# 3. Top 20 Products by Revenue
top_products = df.groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(20)

plt.figure(figsize=(12, 8))
sns.barplot(x=top_products.values, y=top_products.index, palette='magma')
plt.title('Top 20 Products by Revenue', fontsize=14)
plt.xlabel('Total Revenue (£)', fontsize=12)
plt.ylabel('Product Description', fontsize=12)
plt.tight_layout()
plt.show()


**Business Insight:** 
A small subset of products, such as the "REGENCY CAKESTAND 3 TIER" and "WHITE HANGING HEART T-LIGHT HOLDER", drive a disproportionately large share of total revenue. This confirms the Pareto principle (80/20 rule) is at play. The business should ensure these key items are never out of stock and consider bundling them with lower-performing items to increase overall basket size.


In [ ]:
# 4. Revenue by Day of Week
# Map day of week to ensure correct sorting
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_revenue = df.groupby('DayOfWeek')['Revenue'].sum().reindex(days_order)

plt.figure(figsize=(12, 6))
sns.barplot(x=day_revenue.index, y=day_revenue.values, palette='coolwarm')
plt.title('Total Revenue by Day of Week', fontsize=14)
plt.xlabel('Day of Week', fontsize=12)
plt.ylabel('Total Revenue (£)', fontsize=12)
plt.tight_layout()
plt.show()


**Business Insight:** 
Thursdays and Tuesdays/Wednesdays are the peak days for revenue, with very little activity on the weekends (especially Saturday, which appears to have no data or very low data). This is a classic B2B (Business-to-Business) purchasing pattern, where wholesale buyers place orders during the workweek to prepare for their own weekend retail sales. The business should align its customer support and promotional emails to Tuesday-Thursday.


In [ ]:
# 5. Revenue by Hour of Day
hour_revenue = df.groupby('Hour')['Revenue'].sum()

plt.figure(figsize=(12, 6))
sns.barplot(x=hour_revenue.index, y=hour_revenue.values, palette='mako')
plt.title('Total Revenue by Hour of Day', fontsize=14)
plt.xlabel('Hour of Day (24h format)', fontsize=12)
plt.ylabel('Total Revenue (£)', fontsize=12)
plt.tight_layout()
plt.show()


**Business Insight:** 
The vast majority of orders are placed between 10:00 AM and 3:00 PM (15:00), peaking around noon. This further reinforces the B2B customer profile, as purchasing agents are buying during standard office hours. The business should ensure its website servers are scaled to handle peak traffic mid-day and schedule maintenance only during the evening or early morning hours.


In [ ]:
# 6. Cancellation Rate by Month
# Calculate total orders (valid + cancelled) per month
clean_orders = df.groupby([df['InvoiceDate'].dt.year, df['InvoiceDate'].dt.month])['InvoiceNo'].nunique()
cancelled_orders = cancellations_df.groupby([cancellations_df['InvoiceDate'].dt.year, cancellations_df['InvoiceDate'].dt.month])['InvoiceNo'].nunique()

monthly_orders = pd.DataFrame({'Valid': clean_orders, 'Cancelled': cancelled_orders}).fillna(0)
monthly_orders['Total'] = monthly_orders['Valid'] + monthly_orders['Cancelled']
monthly_orders['Cancellation_Rate'] = (monthly_orders['Cancelled'] / monthly_orders['Total']) * 100

# Plotting
monthly_orders = monthly_orders.reset_index()
monthly_orders['Month_Year'] = monthly_orders['level_0'].astype(str) + '-' + monthly_orders['level_1'].astype(str).str.zfill(2)

plt.figure(figsize=(12, 6))
plt.plot(monthly_orders['Month_Year'], monthly_orders['Cancellation_Rate'], marker='o', color='purple', linewidth=2)
plt.title('Cancellation Rate by Month (%)', fontsize=14)
plt.xlabel('Month-Year', fontsize=12)
plt.ylabel('Cancellation Rate (%)', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


**Business Insight:** 
The cancellation rate fluctuates throughout the year but tends to spike alongside high-volume sales months (like late Q4). This suggests that as order volume increases, stock shortages, shipping delays, or buyer's remorse also increase, leading to more returns and cancellations. The operations team needs to improve inventory forecasting and fulfillment efficiency to reduce revenue leakage during peak seasons.


In [ ]:
# 7. UK vs International Revenue Split
df['Month_Year'] = df['InvoiceDate'].dt.to_period('M')

uk_vs_intl = df.groupby(['Month_Year', 'is_uk'])['Revenue'].sum().unstack()
uk_vs_intl.columns = ['International', 'UK']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Pie Chart
total_split = df.groupby('is_uk')['Revenue'].sum()
ax1.pie(total_split, labels=['International', 'UK'], autopct='%1.1f%%', colors=['#ff9999','#66b3ff'], startangle=90)
ax1.set_title('Overall Revenue Split: UK vs International')

# Bar Chart
uk_vs_intl.plot(kind='bar', stacked=False, ax=ax2, color=['#ff9999', '#66b3ff'])
ax2.set_title('Monthly Revenue: UK vs International')
ax2.set_xlabel('Month')
ax2.set_ylabel('Revenue (£)')

plt.tight_layout()
plt.show()


**Business Insight:** 
While the UK constitutes roughly 84% of total revenue over the period, the side-by-side bar chart shows that International revenue is slowly growing and acts as a stable, supplemental income stream. Expanding international shipping options and offering multi-currency checkout could accelerate this growth and diversify the company's risk away from relying solely on the UK economy.


In [ ]:
# 8. Revenue Distribution
plt.figure(figsize=(12, 6))
sns.histplot(df['Revenue'], bins=50, log_scale=(False, True), color='green')
plt.title('Distribution of Transaction Revenue (Log Scale on Y-Axis)', fontsize=14)
plt.xlabel('Revenue per Line Item (£)', fontsize=12)
plt.ylabel('Frequency (Log Scale)', fontsize=12)
plt.tight_layout()
plt.show()


**Business Insight:** 
Most individual transactions have a relatively small revenue value, clustered near the lower end of the spectrum. However, the long tail extending to the right shows there are rare but highly valuable bulk orders. Since acquiring new customers is expensive, identifying these high-value "whale" clients and offering them dedicated account managers or loyalty discounts could secure and increase these massive transactions.


EDA Summary of Business Insights

The 8 charts above reveal the following key patterns in the UCI Online Retail dataset.

Chart 1 - Monthly Revenue Trend
Revenue peaks sharply in November 2011 due to holiday season demand. Q1 is the slowest period of the year.

Chart 2 - Revenue by Country
The UK generates around 84 percent of total revenue. Netherlands and EIRE are the strongest international markets.

Chart 3 - Top 20 Products
A small set of products like REGENCY CAKESTAND 3 TIER drive a disproportionate share of revenue. This confirms the 80/20 Pareto principle.

Chart 4 - Revenue by Day of Week
Thursday and Tuesday are peak days. Saturday has near-zero activity. This is a classic B2B purchasing pattern where buyers order during the workweek.

Chart 5 - Revenue by Hour
10am to 3pm is the peak ordering window. This matches standard business hours and confirms the B2B customer profile.

Chart 6 - Cancellation Rate
Cancellation rate spikes during high-volume months, suggesting fulfilment strain when order volumes are at their peak.

Chart 7 - UK vs International
UK dominates but international is a steady secondary income stream. Netherlands is the strongest growth opportunity.

Chart 8 - Revenue Distribution
Most orders are low value. A small number of bulk orders drive outsized revenue. These bulk buyers are likely wholesale clients.

Overall recommendation: Focus on protecting November to December inventory, reducing peak-season cancellations, and growing Netherlands and EIRE as the top international expansion targets.